# 01 — Retention extraction

Pulls the two base tables for the retention analysis out of the public Firebase/GA4 BigQuery export and writes them to CSV. No analysis happens here.

**Source:** `firebase-public-project.analytics_153293282` — 114 daily event tables, 12 Jun – 3 Oct 2018, 5.7M events, 15,175 distinct `user_pseudo_id`.
**Produces:** `day_0.csv` (earliest install date per user), `user_journal.csv` (one row per user per active day).
**Next:** `02_retention_analysis.ipynb` rebuilds the retention curve from these two files in pandas.

The two cells below authenticate the Colab session and open a BigQuery client. The dataset is public; the project id carries billing and quota only.

In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="flooit-analytics-project")

## Day 0 — first install date per user

Retention needs a per-user clock. `first_open` is the install event, and `MIN` per user collapses anyone with more than one logged `first_open` (three such users, per the Day 4 audit) down to a single Day 0.

The 4,319 users returned are the **cohort**: everyone whose install falls inside the window. The other ~10.8k users installed before 12 Jun and are left-censored — retention is undefined for them, so they are excluded here by construction rather than filtered out downstream.

In [3]:
query_day0 = """
  SELECT user_pseudo_id, MIN(PARSE_DATE('%Y%m%d', event_date)) AS day_0
  FROM `firebase-public-project.analytics_153293282.events_*`
  WHERE event_name = 'first_open'
  GROUP BY user_pseudo_id
"""
day0 = client.query(query_day0).to_dataframe()

In [4]:
print(day0.shape)
print(day0.dtypes)
day0.head()

(4319, 2)
user_pseudo_id    object
day_0             dbdate
dtype: object


,user_pseudo_id,day_0
0,BB752D348C185313C5411520671D1FB4,2018-07-08
1,27458870652D57B3D0B5F1586BDC3E63,2018-07-23
2,678B5107CB9D55A57C122F3A3CE95F51,2018-07-23
3,5879C7CA565C4DFA660DA38E6552E1D5,2018-07-23
4,8E8745CADF1ACC6EE7AADA7A8B8DF1E4,2018-08-01


## User journal — one row per user per active day

`SELECT DISTINCT` on (user, date) collapses everything a user did on a given day into a single row. Retention asks only *"was this user present on day N?"*, so event counts are noise at this grain, and dropping them keeps the extraction thin.

Covers all 15,175 users rather than just the cohort — the pre-window users are needed for the activity-distribution and spike work in notebook 02.

In [5]:
query_user_journal = """
  SELECT DISTINCT user_pseudo_id, PARSE_DATE('%Y%m%d', event_date) AS active_day
  FROM `firebase-public-project.analytics_153293282.events_*`
"""
user_journal = client.query(query_user_journal).to_dataframe()

In [6]:
print(user_journal.shape)
print(user_journal.dtypes)
user_journal.head()

(59037, 2)
user_pseudo_id    object
active_day        dbdate
dtype: object


,user_pseudo_id,active_day
0,4AB0441290F6077849CBE9C74A828F0A,2018-09-11
1,880C220698D2A32CF0709DD95857AB60,2018-09-11
2,F0192BFEC0959BEE0FB12685539F4131,2018-09-11
3,B99FD965B0874192EE94E8A8FE85EA9E,2018-09-11
4,7B12B56BE42BA15EE493BDC9902B54E4,2018-09-11


## Write to CSV

The BigQuery client returns real date types (`dbdate`), so date arithmetic works in this session. CSV is a text format and carries no types: on reload these columns come back as strings. They are re-parsed at the load boundary in notebook 02 — types are handled where data enters or exits a stage, never mid-flow.

In [7]:
day0.to_csv("day_0.csv", index=False)
user_journal.to_csv("user_journal.csv", index=False)

## What this notebook establishes

Two clean base tables: a per-user Day 0 for 4,319 in-window installers, and a 59,037-row activity journal covering all 15,175 users. Both row counts were verified independently in BigQuery and in Colab before saving.

**Not concluded here.** No retention rate is computed — that is notebook 02's job, and its numbers are cross-checked against an independent SQL implementation in `sql/retention_curve.sql`. Two caveats attach to everything downstream: `user_pseudo_id` identifies an app install on a device, not a person, so unique human players number at most 15,175; and pre-window users appear in the journal with their earlier activity invisible, making any lifetime count for that group an undercount.